# F1 Race Position Predictor

A simple machine learning project that predicts a driver's finishing position in a Formula 1 race based on their starting grid position.

**Data:** Official race results from the 2021–2024 F1 seasons, combined into a single dataset.

**Approach:** A Linear Regression model is trained to learn the relationship between starting grid position and final finishing position. It tests the intuition that grid position is a strong predictor of race outcome (though not a perfect one, given overtakes, DNFs, strategy, etc.).

**What this notebook covers:**
1. Loading and cleaning multi-season race data
2. Exploring the relationship between grid and finish position
3. Training/testing a linear regression model
4. Evaluating performance with standard regression metrics (MAE, MSE, RMSE, R²)

## Setup
Importing the libraries needed: `pandas` for data handling, `matplotlib` for visualization, and `scikit-learn` for building and evaluating the linear regression model

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Loading the Data
Reading in F1 race result data for the 2021–2024 seasons from separate CSV files, and tagging each with its `Season` year so they can be identified after merging.

In [ ]:
df_2021 = pd.read_csv("2021.csv")
df_2021["Season"] = 2021
df_2022 = pd.read_csv("2022.csv")
df_2022["Season"] = 2022
df_2023= pd.read_csv("2023.csv")
df_2023["Season"] = 2023
df_2024 = pd.read_csv("2024.csv")
df_2024["Season"] = 2024

## Combining Seasons
Merging all four seasons into a single DataFrame (`f1`) to get one unified dataset to work with.
Choosing four seasons gives us more data which gives a more reliable pattern for our model to learn.

In [ ]:
f1 = pd.concat([df_2021, df_2022, df_2023, df_2024])
print(len(f1))
f1.head()

## Cleaning the Target Column
Converting `Position` to numeric, since some entries (like "DNF" or "DSQ") aren't valid finishing positions. Rows that can't be converted become `NaN` and are dropped.

In [ ]:
f1["Position"] = pd.to_numeric(f1["Position"], errors='coerce')
f1 = f1.dropna(subset=["Position"])
f1["Position"] = f1["Position"].astype(int)
print(len(f1))

## Cleaning the Feature Column
Converting `Starting Grid` to numeric the same way, and checking how many rows have missing/invalid values.

In [ ]:
f1["Starting Grid"] = pd.to_numeric(f1["Starting Grid"], errors='coerce')
f1["Starting Grid"].isnull().sum()

## Dropping Invalid Rows
Removing rows with missing `Starting Grid` values and casting the column to integer type so it's ready for modeling.

In [ ]:
f1 = f1.dropna(subset=["Starting Grid"])
f1["Starting Grid"] = f1["Starting Grid"].astype(int)
print(len(f1))

## Visualizing the Relationship
Plotting `Starting Grid` vs `Position` to visually check if there's a correlation before training a model. This helps us to confirm linear regression is a reasonable choice.

In [ ]:
plt.scatter(f1["Starting Grid"], f1["Position"]) 
plt.xlabel("Starting Grid") 
plt.ylabel("Position") 
plt.title("Visualising The Pattern")
ax=plt.gca()
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
plt.xticks(range(1, 21))
plt.yticks(range(1, 21))
plt.show()

## Defining Features and Target
Setting `Starting Grid` as the input feature (X) and `Position` as the target variable (y) to be predicted.

In [ ]:
X = f1[["Starting Grid"]]
y = f1["Position"]
print(X.shape) 
print(y.shape)

## Train/Test Split
Splitting the data into training (80%) and testing (20%) sets so the model's performance can be evaluated on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print (X_train.shape) 
print (X_test.shape)
print (y_train.shape) 
print (y_test.shape)

## Training the Model
Fitting a Linear Regression model on the training data, and printing the learned coefficient and intercept.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
print(model.coef_)
print(model.intercept_)

## Making Predictions
Using the trained model to predict finishing positions on the test set, and comparing a sample of predictions against the actual values.

In [ ]:
y_pred = model.predict(X_test)
print(y_pred[:10])
print(y_test[:10])

## Evaluating the Model
Calculating standard regression metrics. This includes MAE, MSE, RMSE, and R² that help us to quantify how well the model's predictions match actual results.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("R-squared (R2):", r2)

| Metric | Value |
|--------|-------|
| MAE    | 2.60  |
| RMSE   | 3.37  |
| R²     | 0.57  |


## Predicted vs Actual
Plotting predicted positions against actual positions, with a red reference line (y = x) showing where a perfect prediction would fall.

In [ ]:
plt.scatter(y_test, y_pred)
plt.plot([1, 20], [1, 20], color='red')
plt.xlabel("Actual Position")
plt.ylabel("Predicted Position")
plt.title("Model Predictions")
ax=plt.gca()
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
plt.xticks(range(1, 21))
plt.yticks(range(1, 21))
plt.show()

## 🏁 Conclusion

Its a good model but in the end, it all falls under the unpredictability of Motorsports. This is a model that analyses statistics and makes predictions based on the same. Such a model cannot account for any unexpected mechanical, electrical or engineering failure or delay in the races.

As far as technically concerned, its giving us a pattern, a pattern that is *generally followed, excluding the exceptions*. This model represents how much statistics, realtime data telemetry, newer and faster machine learning models are required in the motorsports field.

>The model's predictions are capped between **3.33** (for a P1 start) and **15.68** (for a P20 start) — meaning it can never predict a podium finish. This isn't due to unpredictability, but the math of the straight line itself: the slope and intercept fix these floor and ceiling values, no matter how well a driver actually performs.

There's several ways to add more parameters for the model to learn from. For example, if we add the weather data of each race according to the tire compound used, the model can analyse and predict at what position a driver will finish based on the tire compound used in that specific weather. There's many ways to add more parameters for the model to have more data to make patterns with.